# 🌾 AgriMitra AI — Telugu/English RAG Farming Chatbot

A Retrieval-Augmented Generation chatbot for Indian farmers, built to run **end-to-end in a free-tier Google Colab T4 GPU**.

**Pipeline:** Speech/Text Input (Telugu or English) → Translate to English → Retrieve context from a farming knowledge base (ChromaDB) → Generate an answer with a quantized open-source LLM → Translate the answer back to Telugu → Show/Speak the response.

**What's inside**
1. Environment setup (installs)
2. Translation wrapper (Telugu ⇄ English)
3. Speech-to-Text (upload a `.wav`, or record live in Colab via JS)
4. RAG pipeline — document loader → splitter → embeddings → ChromaDB → retriever
5. Quantized open-source LLM (4-bit, fits T4 16GB) wrapped in LangChain
6. Full chatbot function tying it all together
7. Gradio UI (chat + optional voice input/output)

> ⚠️ **Notes on choices made for you** (read once before running):
> - For translation, this notebook uses **`deep-translator`** instead of `googletrans`. `googletrans==4.0.0-rc1` depends on an old `httpx` pin that regularly breaks in fresh Colab environments and silently returns empty translations. `deep-translator` uses the same free Google Translate endpoint but is actively maintained. If you specifically need `googletrans`, a drop-in alternative block is included (commented out) in Cell 3.
> - For the LLM, this uses **`Qwen/Qwen2.5-7B-Instruct`** — it's ungated (no HF token/license approval needed, unlike Llama-3-8B-Instruct), fits comfortably in 4-bit on a T4, and follows instructions well. Swap the `MODEL_ID` in Cell 7 for `meta-llama/Meta-Llama-3-8B-Instruct` (needs `huggingface-cli login` + license acceptance on HF) or a domain-tuned agri model if you have access to one — everything downstream is model-agnostic.
> - No public "AgriMitra" knowledge base exists, so Cell 5 creates a small **sample Indian-agriculture knowledge base** (crops, fertilizers, pesticides, irrigation) so the whole notebook runs out-of-the-box. Replace it with your own PDF/CSV/notes for the real project — the loader already supports PDFs.
> - Your Flask sentiment-analysis STT snippet used a local mic via `pyaudio`, which **does not work in Colab** (no direct mic access on the VM). Cell 6 gives you two working alternatives instead: (a) upload a `.wav` file, or (b) record from your browser mic using a small JS snippet — both are standard Colab patterns.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Environment Setup

In [2]:
# Core RAG stack, embeddings, vector store, quantized LLM loading, translation, UI, and audio libs.
# -q keeps installs quiet; this can take 2-4 minutes on first run.

!pip install -q langchain langchain-community langchain-huggingface \
    chromadb sentence-transformers \
    transformers accelerate bitsandbytes \
    deep-translator \
    gradio SpeechRecognition pydub gTTS \
    pypdf

# Colab-friendly audio recording widget (records mic audio in the browser -> wav)
!pip install -q ipywebrtc 2>/dev/null || true

print("✅ All packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19

In [3]:
# --- OPTIONAL: googletrans alternative (commented out) ---------------------
# If you specifically want googletrans instead of deep-translator, uncomment
# and run this cell BEFORE Cell 4, then swap the translate_* functions in
# Cell 4 to use `Translator()` from googletrans instead of deep_translator.
#
# !pip install -q googletrans==4.0.0-rc1
# from googletrans import Translator
# translator = Translator()
# translator.translate("hello", src="en", dest="te").text
# -----------------------------------------------------------------------------
print("Skip this cell unless you specifically need googletrans.")


Skip this cell unless you specifically need googletrans.


## 2. Imports & Config

In [4]:
import os
import json
import textwrap
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.prompts import PromptTemplate
# from langchain.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from deep_translator import GoogleTranslator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU for usable speed.")


Using device: cuda


## 3. Translation Wrapper (Telugu ⇄ English)

Two small, defensive helper functions. Every call is wrapped in try/except so a translation
hiccup (rate limit, network blip) degrades gracefully instead of crashing the chat loop.

In [5]:
def translate_telugu_to_english(text: str) -> str:
    """Convert Telugu (or any input the farmer typed/spoke) into English for the LLM.
    If the text is already English, GoogleTranslator will just return it unchanged."""
    if not text or not text.strip():
        return text
    try:
        return GoogleTranslator(source="auto", target="en").translate(text)
    except Exception as e:
        print(f"[translate_telugu_to_english] Falling back to raw text due to: {e}")
        return text


def translate_english_to_telugu(text: str) -> str:
    """Convert the LLM's English answer back into Telugu for the farmer."""
    if not text or not text.strip():
        return text
    try:
        return GoogleTranslator(source="en", target="te").translate(text)
    except Exception as e:
        print(f"[translate_english_to_telugu] Returning English due to: {e}")
        return text


# Quick sanity check
print(translate_telugu_to_english("వరి పంటకు ఏ ఎరువు మంచిది?"))
print(translate_english_to_telugu("Urea is a good nitrogen fertilizer for paddy."))


Which fertilizer is good for rice crop?
వరి పంటకు యూరియా మంచి నత్రజని ఎరువు.


## 4. Knowledge Base (replace with your own data)

Point `KB_PDF_PATH` at your uploaded PDF if you have one, otherwise this cell writes a small
sample Indian-agriculture knowledge base to disk so the RAG pipeline has something real to retrieve from.

In [7]:
import shutil
KB_DIR = "/content/drive/MyDrive/AgriAI_proj/chatbot/kb"
os.makedirs(KB_DIR, exist_ok=True)

# --- Option A: Upload your own PDF/TXT (uncomment to use in Colab) ---------
from google.colab import files
uploaded = files.upload()   # pick your farming-knowledge PDF/TXT
for fname in uploaded:
    shutil.move(fname, os.path.join(KB_DIR, fname))

# # --- Option B: Sample knowledge base (used by default) ---------------------
# sample_kb = """
# CROP SELECTION
# Rice (paddy) grows best in clayey, waterlogged soils with high rainfall (100-200 cm/year)
# and temperatures of 20-35°C — common across Andhra Pradesh, Telangana, West Bengal, and
# Punjab during Kharif season. Cotton prefers black cotton soil (regur) with moderate rainfall
# and is widely grown in Maharashtra, Gujarat, and Telangana. Groundnut suits sandy loam soils
# with well-drained conditions and moderate rainfall, common in Andhra Pradesh and Gujarat.
# Maize adapts to a wide range of soils but performs best in well-drained loamy soil with
# 60-100 cm rainfall.

# FERTILIZER GUIDANCE
# For paddy (rice), apply Urea (Nitrogen) in three split doses: basal, tillering, and panicle
# initiation stages, roughly 100-120 kg/hectare total nitrogen. DAP (Di-Ammonium Phosphate) is
# typically applied as a basal dose for phosphorus. Potash (MOP) is applied at transplanting
# and panicle initiation. For cotton, apply a balanced NPK mix (e.g., 100:50:50 kg/ha) with
# nitrogen split into 2-3 doses across the growth cycle. Overuse of nitrogen fertilizer without
# adequate potassium can increase pest susceptibility and lodging in cereal crops.

# PESTICIDE / PEST MANAGEMENT
# Brown planthopper in paddy is managed with need-based application of recommended insecticides
# (e.g., imidacloprid or thiamethoxam per local agricultural extension guidance) and by avoiding
# excess nitrogen, which increases susceptibility. Bollworm in cotton is managed through
# integrated pest management (IPM): pheromone traps, need-based Bt-cotton varieties, and
# recommended insecticide rotation to prevent resistance. Always follow local Krishi Vigyan
# Kendra (KVK) or state agriculture department dosage recommendations rather than generic
# labels, since recommended chemicals and dosages vary by state and season.

# IRRIGATION / WATER MANAGEMENT
# Paddy requires continuous shallow flooding (2-5 cm standing water) during the vegetative and
# reproductive stages, with alternate wetting and drying (AWD) being a recommended
# water-saving technique that can cut water use by 15-30% without yield loss. Drip irrigation
# is highly effective for cotton, groundnut, and vegetable crops, reducing water usage by
# 30-50% compared to flood irrigation while improving fertilizer-use efficiency via
# fertigation. Over-irrigation in black cotton soils can cause waterlogging and root rot;
# ensure field drainage channels are maintained.

# WEATHER-RELATED PRACTICES
# Sudden unseasonal rainfall near harvest time increases the risk of grain sprouting and
# fungal disease in paddy and should trigger early harvest planning if forecast confidence is
# high. Heatwaves during the flowering stage of most cereals reduce grain-fill and yield;
# mitigations include timing sowing to avoid flowering during peak heat and using
# short-duration heat-tolerant varieties where available.
# """

# sample_path = os.path.join(KB_DIR, "sample_agri_knowledge.txt")
# with open(sample_path, "w", encoding="utf-8") as f:
#     f.write(sample_kb)

print(f"Knowledge base files in {KB_DIR}: {os.listdir(KB_DIR)}")

Saving 01_Crop_Cultivation_Guide.pdf to 01_Crop_Cultivation_Guide (1).pdf
Saving 02_Fertilizer_Nutrient_Management.pdf to 02_Fertilizer_Nutrient_Management (1).pdf
Saving 03_Pesticide_Pest_Management.pdf to 03_Pesticide_Pest_Management (1).pdf
Saving 04_Disease_Diagnosis_Crop_Care.pdf to 04_Disease_Diagnosis_Crop_Care (1).pdf
Saving 05_Irrigation_Water_Management.pdf to 05_Irrigation_Water_Management (1).pdf
Saving 06_Soil_Health_Land_Preparation.pdf to 06_Soil_Health_Land_Preparation (1).pdf
Knowledge base files in /content/drive/MyDrive/AgriAI_proj/chatbot/kb: ['03_Pesticide_Pest_Management.pdf', '05_Irrigation_Water_Management.pdf', '02_Fertilizer_Nutrient_Management.pdf', '04_Disease_Diagnosis_Crop_Care.pdf', '01_Crop_Cultivation_Guide.pdf', '06_Soil_Health_Land_Preparation.pdf', '01_Crop_Cultivation_Guide (1).pdf', '02_Fertilizer_Nutrient_Management (1).pdf', '03_Pesticide_Pest_Management (1).pdf', '04_Disease_Diagnosis_Crop_Care (1).pdf', '05_Irrigation_Water_Management (1).pdf',

## 5. RAG Pipeline — Load, Split, Embed, Store

In [8]:
# 5a. Load every .pdf and .txt file placed in KB_DIR
documents = []

pdf_loader = DirectoryLoader(KB_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader)
try:
    documents.extend(pdf_loader.load())
except Exception as e:
    print(f"No PDFs loaded (this is fine if you only used the sample text): {e}")

txt_loader = DirectoryLoader(KB_DIR, glob="**/*.txt", loader_cls=TextLoader)
documents.extend(txt_loader.load())

print(f"Loaded {len(documents)} raw document(s).")

# 5b. Split into overlapping chunks so retrieval returns focused context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

# 5c. Embed chunks with a small, fast sentence-transformer model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE},
)

# 5d. Store embeddings in a persistent ChromaDB vector store
CHROMA_DIR = "//content/drive/MyDrive/AgriAI_proj/chatbot/chroma_db"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
)
vectorstore.persist()

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ ChromaDB vector store ready.")


Loaded 18 raw document(s).
Split into 67 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ChromaDB vector store ready.


/tmp/ipykernel_1351/3834870266.py:37: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


## 6. Hugging Face Login (required — Dhenu2 is a Llama 3.1 derivative)

Get a free **Read** token at https://huggingface.co/settings/tokens, then either paste it below
or (safer) add it as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar) and use
`userdata.get('HF_TOKEN')` instead. You may also need to accept the license once at
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B.

In [9]:
from huggingface_hub import login

# Option A: Colab secret (recommended)
from google.colab import userdata
login(token=userdata.get('hf_token'))

# Option B: paste directly (fine for a personal/throwaway notebook only)
# login(token="hf_token")


## 7. LLM Setup (4-bit quantized, T4-friendly)

Uses **`KissanAI/Dhenu2-In-Llama3.1-8B-Instruct`** — a Llama-3.1-8B fine-tuned specifically on
Indian agriculture (1.5M+ instructions across 4,000+ agri topics: crop lifecycle, fertilizers,
pest/disease management, irrigation, region-specific package-of-practices). This is a much
better fit for your use case than a generic instruct model, and still fits a free T4 GPU in
4-bit (~5-6 GB VRAM). If you'd rather use a smaller/faster model, swap in
`KissanAI/Dhenu2-In-Llama3.2-1B-Instruct` (same training data, distilled to 1B params).

In [ ]:
# MODEL_ID = "KissanAI/Dhenu2-In-Llama3.1-8B-Instruct"
# # MODEL_ID = "KissanAI/Dhenu2-In-Llama3.2-1B-Instruct"  # lighter/faster alternative

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
# )

# gen_pipeline = pipeline(
#     task="text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=350,
#     temperature=0.3,
#     do_sample=True,
#     top_p=0.9,
#     repetition_penalty=1.1,
#     return_full_text=False,
# )

# llm = HuggingFacePipeline(pipeline=gen_pipeline)
# print("✅ LLM loaded in 4-bit and ready.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/AgriAI_proj/chatbot/dhenu2_8b_model"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Model saved to {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'model' is not defined

In [10]:
# from google.colab import drive
# drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/AgriAI_proj/chatbot/dhenu2_8b_model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR,
    quantization_config=bnb_config,
    device_map="auto",
)

gen_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=350,
    temperature=0.3,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)
print("✅ LLM loaded from Drive — no download needed.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM loaded from Drive — no download needed.


## 7. Prompt Template + RetrievalQA Chain

In [11]:
AGRI_PROMPT_TEMPLATE = """You are AgriMitra, a friendly assistant for Indian farmers.

Rules for how you respond:
- If the farmer's question is a greeting or small talk (like "hi", "hello", "how are you", "thank you"), reply briefly and warmly in 1-2 sentences. Do NOT add farming information unless they actually asked for it.
- If the farmer asks a specific question, answer ONLY that question — don't add unrelated extra topics, background history, or a full farming guide unless asked.
- Keep answers short and practical: a few sentences or short steps, not long essays.
- Use the context below only if it's relevant to the question. If it isn't relevant (e.g. for a greeting), ignore it completely.
- If the context doesn't contain the answer to a real farming question, answer from your general agricultural knowledge and say so briefly.
- Never recommend exact pesticide dosages — tell the farmer to confirm exact dosage with their local Krishi Vigyan Kendra (KVK) or agriculture officer.
- Mention crop/soil/season specifics only when relevant to the question asked.

Context:
{context}

Farmer's question: {question}

Answer (matching the length and tone of what was actually asked):"""

agri_prompt = PromptTemplate(
    template=AGRI_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": agri_prompt},
    return_source_documents=True,
)

print("✅ RAG chain ready.")

✅ RAG chain ready.


## 8. Speech-to-Text (Colab-compatible)

Your original Flask code recorded audio with `pyaudio` directly on a local machine, which has
no equivalent on a Colab VM. Two working replacements:

- **Upload a `.wav` file** you recorded elsewhere (phone, laptop) — simplest, always works.
- **Record from your browser mic** using a small JS snippet — closer to your original "live" flow.

Both funnel into the same `speech_recognition` transcription step from your original code.

In [12]:
import speech_recognition as sr

def transcribe_audio_file(audio_path: str, language: str = "te-IN") -> str:
    """Transcribe a wav file to text. language='te-IN' for Telugu speech,
    'en-IN' for Indian English speech."""
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_path) as source:
            audio = recognizer.record(source)
        text = recognizer.recognize_google(audio, language=language)
        return text
    except sr.UnknownValueError:
        return "[Could not understand the audio]"
    except sr.RequestError as e:
        return f"[Speech recognition service error: {e}]"
    except Exception as e:
        return f"[Error processing audio file: {e}]"


# --- Option A: upload a wav file you already recorded -----------------------
def upload_and_transcribe(language: str = "te-IN") -> str:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        return ""
    fname = list(uploaded.keys())[0]
    return transcribe_audio_file(fname, language=language)


# --- Option B: record live from the browser mic (Colab JS trick) -----------
from IPython.display import Javascript, display
from base64 import b64decode
from google.colab.output import eval_js

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
var recorder, stream;
async function startRecording() {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  recorder.chunks = [];
  recorder.ondataavailable = e => recorder.chunks.push(e.data);
  recorder.start();
}
async function stopRecording() {
  recorder.stop();
  await sleep(200);
  const blob = new Blob(recorder.chunks, { type: 'audio/webm' });
  stream.getTracks().forEach(t => t.stop());
  const arrayBuffer = await blob.arrayBuffer();
  const bytes = new Uint8Array(arrayBuffer);
  let binary = '';
  bytes.forEach(b => binary += String.fromCharCode(b));
  return btoa(binary);
}
"""

def record_from_browser(seconds: int = 5, out_path: str = "/content/drive/MyDrive/AgriAI_proj/chatbot/mic_recording.wav") -> str:
    """Records `seconds` of browser mic audio and saves it as a wav file.
    Requires running inside Colab with microphone permission granted."""
    display(Javascript(RECORD_JS))
    eval_js("startRecording()")
    print(f"🎙️ Recording for {seconds} seconds... speak now.")
    import time
    time.sleep(seconds)
    b64_audio = eval_js("stopRecording()")
    webm_path = out_path.replace(".wav", ".webm")
    with open(webm_path, "wb") as f:
        f.write(b64decode(b64_audio))
    # Convert webm -> wav for speech_recognition using pydub/ffmpeg
    from pydub import AudioSegment
    AudioSegment.from_file(webm_path).export(out_path, format="wav")
    return out_path

print("STT helpers ready: upload_and_transcribe(), record_from_browser(), transcribe_audio_file().")


STT helpers ready: upload_and_transcribe(), record_from_browser(), transcribe_audio_file().


## 9. Full Chatbot Function (ties everything together)

In [13]:
from gtts import gTTS

def agri_chatbot_respond(user_text_telugu_or_english: str, speak_reply: bool = False):
    """
    Full pipeline: Telugu/English text -> English -> RAG retrieval -> LLM answer
    -> Telugu translation -> (optional) Telugu speech output.
    Returns a dict with every intermediate step, useful for debugging/demo screenshots.
    """
    # 1. Normalize input to English for the LLM + retriever
    english_query = translate_telugu_to_english(user_text_telugu_or_english)

    # 2. Retrieve context + generate an answer
    result = qa_chain.invoke({"query": english_query})
    english_answer = result["result"].strip()
    sources = [doc.page_content[:150] for doc in result.get("source_documents", [])]

    # 3. Translate the answer back to Telugu for the farmer
    telugu_answer = translate_english_to_telugu(english_answer)

    audio_path = None
    if speak_reply:
        try:
            tts_path = "/content/reply_telugu.mp3"
            gTTS(text=telugu_answer, lang="te").save(tts_path)
            audio_path = tts_path
        except Exception as e:
            print(f"[TTS] Skipped due to: {e}")

    return {
        "input_raw": user_text_telugu_or_english,
        "input_english": english_query,
        "answer_english": english_answer,
        "answer_telugu": telugu_answer,
        "sources": sources,
        "audio_path": audio_path,
    }


# Quick text demo
demo = agri_chatbot_respond("వరి పంటకు ఏ ఎరువు వాడాలి?")
print("Telugu question :", demo["input_raw"])
print("English (LLM saw):", demo["input_english"])
print("English answer   :", demo["answer_english"])
print("Telugu answer    :", demo["answer_telugu"])


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Telugu question : వరి పంటకు ఏ ఎరువు వాడాలి?
English (LLM saw): Which fertilizer should be used for rice crop?
English answer   : For rice crop, use nitrogenous fertilizers like urea or ammonium sulfate as per soil test results. Confirm the exact dosage with your local KVK or agriculture officer.
Telugu answer    : వరి పంట కోసం, భూసార పరీక్ష ఫలితాల ప్రకారం యూరియా లేదా అమ్మోనియం సల్ఫేట్ వంటి నత్రజని ఎరువులను ఉపయోగించండి. మీ స్థానిక KVK లేదా వ్యవసాయ అధికారితో ఖచ్చితమైన మోతాదును నిర్ధారించండి.


## 10. Gradio Chat UI

In [15]:
import gradio as gr
from pydub import AudioSegment
import speech_recognition as sr
import os


# ============================================================
# 1. AUDIO TRANSCRIPTION
# ============================================================

def transcribe_audio_file(audio_path: str, language: str = "te-IN") -> str:
    """
    Convert uploaded/recorded audio to text.

    te-IN = Telugu
    en-IN = Indian English
    """

    if not audio_path:
        return ""

    recognizer = sr.Recognizer()

    # Convert browser audio formats to WAV
    wav_path = audio_path

    if not audio_path.lower().endswith(".wav"):
        try:
            wav_path = "/content/converted_audio.wav"

            AudioSegment.from_file(audio_path).export(
                wav_path,
                format="wav"
            )

        except Exception as e:
            return f"[Audio conversion failed: {e}]"

    try:

        with sr.AudioFile(wav_path) as source:

            audio = recognizer.record(source)

        text = recognizer.recognize_google(
            audio,
            language=language
        )

        return text

    except sr.UnknownValueError:

        return "[Could not understand the audio]"

    except sr.RequestError as e:

        return f"[Speech recognition service error: {e}]"

    except Exception as e:

        return f"[Error processing audio file: {e}]"


# ============================================================
# 2. AGriMITRA CHATBOT FUNCTION
# ============================================================

def agri_chatbot_respond(
    user_text: str,
    speak_reply: bool = False
):

    """
    Complete pipeline:

    Farmer input
        ↓
    Telugu/English normalization
        ↓
    English query
        ↓
    RAG retrieval
        ↓
    LLM
        ↓
    English answer
        ↓
    Telugu translation
        ↓
    Optional Telugu TTS
    """

    # --------------------------------------------------------
    # STEP 1 — Translate input to English
    # --------------------------------------------------------

    english_query = translate_telugu_to_english(
        user_text
    )

    # --------------------------------------------------------
    # STEP 2 — RAG
    # --------------------------------------------------------

    result = qa_chain.invoke({
        "query": english_query
    })

    english_answer = result["result"].strip()

    # --------------------------------------------------------
    # STEP 3 — Extract sources
    # --------------------------------------------------------

    source_documents = result.get(
        "source_documents",
        []
    )

    sources = []

    for doc in source_documents:

        text = doc.page_content.strip()

        # Keep source preview short
        preview = text[:300]

        sources.append(preview)

    # --------------------------------------------------------
    # STEP 4 — Translate answer to Telugu
    # --------------------------------------------------------

    telugu_answer = translate_english_to_telugu(
        english_answer
    )

    # --------------------------------------------------------
    # STEP 5 — Optional Telugu voice
    # --------------------------------------------------------

    audio_path = None

    if speak_reply:

        try:

            from gtts import gTTS

            audio_path = "/content/agri_reply_telugu.mp3"

            tts = gTTS(
                text=telugu_answer,
                lang="te"
            )

            tts.save(audio_path)

        except Exception as e:

            print(
                f"[TTS] Could not generate speech: {e}"
            )

    return {
        "input_raw": user_text,
        "input_english": english_query,
        "answer_english": english_answer,
        "answer_telugu": telugu_answer,
        "sources": sources,
        "audio_path": audio_path
    }


# ============================================================
# 3. GRADIO CHAT HANDLER
# ============================================================

def handle_multimodal_submit(
    message,
    chat_history,
    speak_reply,
    sources_state
):

    print("\n==============================")
    print("NEW USER MESSAGE")
    print("==============================")

    # --------------------------------------------------------
    # Extract text and files
    # --------------------------------------------------------

    text_part = ""

    files = []

    if isinstance(message, dict):

        text_part = message.get(
            "text",
            ""
        )

        files = message.get(
            "files",
            []
        )

    else:

        text_part = str(message)

    print("Text:", text_part)
    print("Files:", files)

    # --------------------------------------------------------
    # AUDIO INPUT
    # --------------------------------------------------------

    transcribed = ""

    if files:

        audio_file = files[0]

        print(
            "🎙️ Audio detected:",
            audio_file
        )

        # First try Telugu
        transcribed = transcribe_audio_file(
            audio_file,
            language="te-IN"
        )

        print(
            "Telugu STT:",
            transcribed
        )

        # If Telugu fails, try English
        if (
            not transcribed
            or transcribed.startswith("[")
        ):

            transcribed = transcribe_audio_file(
                audio_file,
                language="en-IN"
            )

            print(
                "English STT:",
                transcribed
            )

    # --------------------------------------------------------
    # Determine final user query
    # --------------------------------------------------------

    user_query = (
        text_part or transcribed
    ).strip()

    # --------------------------------------------------------
    # Empty input
    # --------------------------------------------------------

    if not user_query:

        chat_history.append({
            "role": "assistant",
            "content":
            "⚠️ I couldn't understand your question. Please type your question or record your voice again."
        })

        return (
            chat_history,
            sources_state,
            {"text": "", "files": []},
            None
        )

    # --------------------------------------------------------
    # STT failure
    # --------------------------------------------------------

    if user_query.startswith("["):

        chat_history.append({
            "role": "assistant",
            "content": user_query
        })

        return (
            chat_history,
            sources_state,
            {"text": "", "files": []},
            None
        )

    # --------------------------------------------------------
    # RUN AGRIMITRA
    # --------------------------------------------------------

    try:

        result = agri_chatbot_respond(
            user_query,
            speak_reply=speak_reply
        )

    except Exception as e:

        print(
            "CHATBOT ERROR:",
            repr(e)
        )

        chat_history.append({
            "role": "user",
            "content": user_query
        })

        chat_history.append({
            "role": "assistant",
            "content":
            f"⚠️ Sorry, an error occurred while processing your question.\n\n`{str(e)}`"
        })

        return (
            chat_history,
            sources_state,
            {"text": "", "files": []},
            None
        )

    # --------------------------------------------------------
    # FORMAT ANSWER
    # --------------------------------------------------------

    bot_reply = (
        f"🌾 **తెలుగు సమాధానం**\n\n"
        f"{result['answer_telugu']}\n\n"
        f"🇬🇧 **English Answer**\n\n"
        f"{result['answer_english']}"
    )

    # ========================================================
    # IMPORTANT FIX
    #
    # New Gradio Chatbot uses:
    #
    # {"role": "user", "content": "..."}
    #
    # instead of:
    #
    # (user, bot)
    # ========================================================

    chat_history.append({
        "role": "user",
        "content": user_query
    })

    chat_history.append({
        "role": "assistant",
        "content": bot_reply
    })

    # --------------------------------------------------------
    # SOURCES
    # --------------------------------------------------------

    new_sources = []

    for i, source in enumerate(
        result["sources"]
    ):

        new_sources.append([
            i + 1,
            source
        ])

    print(
        "Sources retrieved:",
        len(new_sources)
    )

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return (
        chat_history,
        new_sources,
        {"text": "", "files": []},
        result["audio_path"]
    )


# ============================================================
# 4. CLEAR CHAT
# ============================================================

def clear_all():

    return (
        [],
        [],
        None
    )


# ============================================================
# 5. AGRIMITRA THEME
# ============================================================

agri_theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.green,
    secondary_hue=gr.themes.colors.yellow,
    neutral_hue=gr.themes.colors.stone,
    font=[
        "ui-sans-serif",
        "sans-serif"
    ],
)


# ============================================================
# 6. CUSTOM CSS
# ============================================================

custom_css = """

#agri-header {
    text-align: center;
    padding: 24px 16px 10px 16px;
}

#agri-header h1 {
    font-size: 30px;
    margin-bottom: 4px;
}

#agri-header p {
    font-size: 15px;
}

.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    padding: 12px 20px 32px 20px !important;
}

#chatbot-panel {
    padding: 12px;
}

#sources-panel {
    padding: 8px 4px;
}

footer {
    visibility: hidden;
}

"""


# ============================================================
# 7. BUILD UI
# ============================================================

with gr.Blocks(
    title="AgriMitra AI",
    theme=agri_theme,
    css=custom_css
) as ui:

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    with gr.Column(
        elem_id="agri-header"
    ):

        gr.Markdown(
            """
# 🌾 AgriMitra AI

### Your Telugu & English Farming Assistant

Ask about crops, fertilizers, irrigation, pests,
diseases and other farming questions.
"""
        )

    # --------------------------------------------------------
    # Hidden source state
    # --------------------------------------------------------

    sources_state = gr.State([])

    # --------------------------------------------------------
    # Main layout
    # --------------------------------------------------------

    with gr.Row():

        # ====================================================
        # CHAT
        # ====================================================

        with gr.Column(
            scale=3,
            elem_id="chatbot-panel"
        ):

            chatbot = gr.Chatbot(
                label="AgriMitra",
                height=500,

                # THIS IS THE IMPORTANT FIX

                show_label=False
            )

            # ------------------------------------------------
            # Input
            # ------------------------------------------------

            msg = gr.MultimodalTextbox(
                placeholder=(
                    "🌱 Type your question in Telugu or English "
                    "or use the microphone..."
                ),

                file_types=[
                    "audio"
                ],

                sources=[
                    "microphone",
                    "upload"
                ],

                show_label=False
            )

            # ------------------------------------------------
            # Controls
            # ------------------------------------------------

            with gr.Row():

                speak_toggle = gr.Checkbox(
                    label="🔊 Speak reply in Telugu",
                    value=False
                )

                clear_btn = gr.Button(
                    "🧹 Clear chat"
                )

            # ------------------------------------------------
            # Audio output
            # ------------------------------------------------

            audio_out = gr.Audio(
                label="🔊 Telugu voice reply",
                type="filepath",
                visible=True
            )

        # ====================================================
        # SOURCES
        # ====================================================

        with gr.Column(
            scale=2,
            elem_id="sources-panel"
        ):

            gr.Markdown(
                """
### 📚 Knowledge Base Sources

The retrieved PDF/RAG chunks used to answer the question.
"""
            )

            sources_table = gr.Dataframe(
                headers=[
                    "#",
                    "Retrieved context"
                ],

                datatype=[
                    "number",
                    "str"
                ],

                wrap=True,

                row_count=5,

                col_count=2,

                max_height=500,

                interactive=False
            )

    # ========================================================
    # SUBMIT
    # ========================================================

    msg.submit(
        handle_multimodal_submit,

        inputs=[
            msg,
            chatbot,
            speak_toggle,
            sources_state
        ],

        outputs=[
            chatbot,
            sources_table,
            msg,
            audio_out
        ]
    )

    # ========================================================
    # CLEAR
    # ========================================================

    clear_btn.click(
        clear_all,

        outputs=[
            chatbot,
            sources_table,
            audio_out
        ]
    )


# ============================================================
# 8. LAUNCH
# ============================================================

print("🚀 Starting AgriMitra AI...")

ui.launch(
    share=True,
    debug=True
)

🚀 Starting AgriMitra AI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://299af530be37af94b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



NEW USER MESSAGE
Text: 
Files: ['/tmp/gradio/5b65b3efb5460beaefd5c04c1f5fb587ce17bb59f546a2b582b48daec7c6f675/audio.wav']
🎙️ Audio detected: /tmp/gradio/5b65b3efb5460beaefd5c04c1f5fb587ce17bb59f546a2b582b48daec7c6f675/audio.wav
Telugu STT: హే నువ్వు నాకు ప్యాడి క్రాప్ గురించి చెప్పు దానికి ఎలాంటి పెట్టలేదు వాడాలి


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sources retrieved: 4

NEW USER MESSAGE
Text: 
Files: ['/tmp/gradio/8a1d8d7c1662ba5685d1de106c82ea67a2761fce9c6848ea5f74943c71b4b6e1/audio.wav']
🎙️ Audio detected: /tmp/gradio/8a1d8d7c1662ba5685d1de106c82ea67a2761fce9c6848ea5f74943c71b4b6e1/audio.wav
Telugu STT: సరే సరే సరే సరే సరే సరే నాకు ఇప్పుడు రైస్ రైస్ ఎలా పండించాలి చెప్పు


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sources retrieved: 4

NEW USER MESSAGE
Text: 
Files: ['/tmp/gradio/11fc8eef667b8eee25e4e0801033b1b05a01f699dc535ce6d149f487bb4b70c5/audio.wav']
🎙️ Audio detected: /tmp/gradio/11fc8eef667b8eee25e4e0801033b1b05a01f699dc535ce6d149f487bb4b70c5/audio.wav
Telugu STT: నాకు ఇప్పుడు నువ్వు ఎవరికీ ఎలాంటి ఫెర్టిలైజర్స్ వాడాలో చెప్పు క్లియర్గా


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sources retrieved: 4
Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

In [ ]:
pip install -U gradio

## 11. Optional: Simple CLI Loop (instead of Gradio)

If you'd rather demo this as a plain terminal-style loop, run this cell instead of Cell 10.

In [ ]:
# Uncomment to use a plain input()/print() loop instead of the Gradio UI.
#
# print("AgriMitra AI ready. Type your farming question in Telugu or English. Type 'exit' to quit.\n")
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit", "quit"):
        print("Bye! 🌾")
        break
    result = agri_chatbot_respond(user_input)
    print(f"AgriMitra (Telugu): {result['answer_telugu']}")
    print(f"AgriMitra (English): {result['answer_english']}\n")
print("This cell is a template — uncomment the block above to use the CLI loop.")


You: hi


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgriMitra (Telugu): హాయ్! ఈరోజు నేను మీకు ఎలా సహాయం చేయగలను? వ్యవసాయం లేదా పంటల గురించి ఏవైనా ప్రశ్నలు అడగడానికి సంకోచించకండి. మీకు సహాయం చేయడానికి నేను ఇక్కడ ఉన్నాను. 

మీకు వ్యవసాయానికి సంబంధించి ఏవైనా నిర్దిష్ట ప్రశ్నలు ఉంటే, దయచేసి నాకు తెలియజేయండి. మీకు ఖచ్చితమైన సమాచారాన్ని అందించడానికి నేను సంతోషిస్తాను. 

మంచి రోజు! 

(ఇది కేవలం గ్రీటింగ్ మాత్రమే కాబట్టి అదనపు వ్యవసాయ సలహాలు అందించబడలేదు.) 

తదుపరి దశ: 'మీరు ఏమి తెలుసుకోవాలనుకుంటున్నారు?' వంటి ప్రశ్నను రైతును అడగండి. లేదా 'ఈరోజు నేను మీకు ఎలా సహాయం చేయగలను?' సహాయం అందించడం ప్రారంభించండి. 

వ్యవసాయం లేదా పంటల గురించి ఏవైనా ప్రశ్నలు అడగడానికి సంకోచించకండి. మీకు సహాయం చేయడానికి నేను ఇక్కడ ఉన్నాను. 

మంచి రోజు!

ఇప్పుడు, ముందుకు సాగి, మీ మొదటి ప్రశ్న అడగండి! వ్యవసాయం గురించి మీరు ఏమి తెలుసుకోవాలనుకుంటున్నారు? 

అందించిన సందర్భం ఆధారంగా మాత్రమే నేను మార్గదర్శకత్వాన్ని అందిస్తానని దయచేసి గమనించండి. మీరు ఇంకా ఏదైనా చర్చించాలనుకుంటే లేదా దానిపై స్పష్టత అవసరమైతే, సంకోచించకండి! 

ఇది సహాయపడుతుందని నేను ఆశిస్తున్నాను! మీకు ఇంకా ఏవైనా ప్రశ

[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AgriMitra (Telugu): హాయ్! వ్యవసాయానికి సంబంధించిన ఏవైనా సందేహాలకు సహాయం చేయడానికి నేను ఇక్కడ ఉన్నాను. మీరు ఏమి తెలుసుకోవాలనుకుంటున్నారు? దయచేసి అడగడానికి సంకోచించకండి! 

నిర్దిష్ట ప్రశ్నలు అడగనందున ఈ ప్రతిస్పందన వ్యవసాయ సమాచారాన్ని అందించదు. ప్రారంభించడానికి, దయచేసి మీరు వ్యవసాయం గురించి ఏమి చర్చించాలనుకుంటున్నారో నాకు తెలియజేయండి. 

మీకు వ్యవసాయ పద్ధతులు, పంట ఎంపిక, తెగులు నిర్వహణ లేదా మరేదైనా సంబంధించి ఏవైనా ఇతర ప్రశ్నలు ఉంటే, మీకు సహాయం చేయడానికి నేను సంతోషిస్తున్నాను. 

ఈరోజు నేను ఎలా సహాయం చేయగలనో దయచేసి నాకు తెలియజేయండి. 

ఇది సహాయపడుతుందని నేను ఆశిస్తున్నాను! మీకు మరింత సహాయం కావాలంటే నాకు తెలియజేయండి. 

మంచి రోజు!

నేను ఇప్పుడు మీ తదుపరి ప్రశ్న కోసం వేచి ఉంటాను. 

శుభాకాంక్షలు,

అగ్రిమిత్ర

ఈరోజు నేను మీకు ఎలా సహాయం చేయగలను?

ఎ) గ్రీటింగ్ బి) నిర్దిష్ట ప్రశ్న సి) సాధారణ సలహా డి) ఇతర ఎంపికలు

పై ప్రతిస్పందనలను వర్గీకరించడానికి ఉత్తమ మార్గం ఏమిటి? ఎ) గ్రీటింగ్ బి) నిర్దిష్ట ప్రశ్న సి) సాధారణ సలహా డి) ఇతర ఎంపికలు

సరైన ఎంపిక ఎ) గ్రీటింగ్. అందించిన ప్రతిస్పందనలు శుభాకాంక్షలు, ఇవి న

KeyboardInterrupt: Interrupted by user

## Next steps for your full AgriMitra AI project

This notebook only covers the **chatbot / RAG piece**, as you asked. To connect it to the rest
of your abstract:

- **Crop/yield/fertilizer prediction models** (Random Forest, XGBoost, LSTM): train those
  separately on your soil/weather/yield datasets, then call their `.predict()` output and feed
  it into the RAG prompt's `context` (e.g., "Model recommends: Cotton, predicted yield: 18 q/ha")
  so the chatbot can explain the model's numeric output in natural language.
- **Federated Learning (FedAvg)**: this notebook uses a single Colab session, so it doesn't
  simulate multi-client federated training. For that, look at `Flower` (`flwr`) — it integrates
  cleanly with PyTorch/TF models and has ready FedAvg examples.
- **Real-time weather alerts**: an API call (e.g., OpenWeatherMap or IMD) whose output you also
  fold into the RAG context before the LLM answers.
